# Imports

In [1]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
import optuna
import shap

import umap
import hdbscan

from joblib import dump, load
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, recall_score

/home/linkezio/Projects/A Hierarchical MEC and IoT Solution for Cyber-Attack Detection in 6G Networks/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Constants

In [2]:
RANDOM_SEED = 42

# Inputs

In [3]:
DB_PATH = "../data/sqlite/data.db"
TABLE   = "CIC_IIoT_dataset_2025"

# Samples per class (defined by the `label` column).
# - int  → same cap for every class
# - None → all available data (careful: may crash on large datasets)
SAMPLES_PER_CLASS: int | None = 10_000
# Per-class overrides — take precedence over SAMPLES_PER_CLASS.
SAMPLES_PER_CLASS_OVERRIDE: dict = {}  # e.g. {"benign": 20_000}

# Data

## Load Data

In [4]:
def load_dataset_from_db(
    db_path: str,
    table: str,
    samples_per_class: "int | None" = None,
    override: dict = {},
) -> pd.DataFrame:
    """
    Load data from SQLite, sampling per class directly in the DB so that
    only the selected rows are pulled into memory.

    SQLite handles sampling with ORDER BY RANDOM() LIMIT n, avoiding the
    need to load the full dataset into pandas before filtering.
    """
    conn = sqlite3.connect(db_path)

    classes = pd.read_sql(
        f'SELECT DISTINCT label FROM "{table}" WHERE label IS NOT NULL', conn
    )["label"].tolist()
    print(f"Classes: {sorted(classes)}")

    frames = []
    for cls in classes:
        n = override.get(cls, samples_per_class)

        if n is None:
            query = f'SELECT * FROM "{table}" WHERE label = ?'
            df_cls = pd.read_sql(query, conn, params=(cls,))
        else:
            query = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
            df_cls = pd.read_sql(query, conn, params=(cls, n))

        frames.append(df_cls)
        print(f"  {cls}: {len(df_cls):,} rows")

    conn.close()

    result = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=42)
    print(f"\nTotal: {len(result):,} rows")
    
    return result

In [5]:
def load_dataset_from_csv(
    data_root: str,
    dataset: str,
    samples_per_class: "int | None" = None,
    override: dict = {},
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Alternative: load from CSV files directly.
    Only use for small datasets — large ones will exhaust RAM.
    """
    import glob
    dataset_path = os.path.join(data_root, dataset)
    csv_paths = sorted(glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True))
    if not csv_paths:
        raise FileNotFoundError(f"No CSVs found in: {dataset_path}")

    df = pd.concat((pd.read_csv(p) for p in csv_paths), ignore_index=True)

    if samples_per_class is None and not override:
        return df

    sampled = []
    for cls in df["label"].unique():
        n = override.get(cls, samples_per_class)
        cls_df = df[df["label"] == cls]
        sampled.append(cls_df.sample(min(n, len(cls_df)), random_state=random_state) if n else cls_df)

    return pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=random_state)

In [6]:
df = load_dataset_from_db(
    DB_PATH,
    TABLE,
    samples_per_class=SAMPLES_PER_CLASS,
    override=SAMPLES_PER_CLASS_OVERRIDE,
)
df.head()

Classes: ['benign', 'bruteforce', 'ddos', 'dos', 'malware', 'mitm', 'web']
  benign: 10,000 rows
  bruteforce: 6,667 rows
  ddos: 10,000 rows
  dos: 10,000 rows
  malware: 10,000 rows
  mitm: 5,466 rows
  web: 10,000 rows

Total: 62,133 rows


,src_ip,dst_ip,src_port,dst_port,protocol,timestamp,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
49942,192.168.1.1,192.168.1.255,45898,20002,17,1.737996e+09,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,mitm
54998,192.168.1.100,192.168.1.195,60992,4280,6,1.738096e+09,0.036475,163453.607133,383.822627,191.911313,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,web
43435,192.168.1.193,192.168.1.80,35586,23,6,1.749752e+09,0.003030,33000.031471,660.000629,330.000315,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware
22839,192.168.1.102,192.168.1.19,2444,1883,6,1.737817e+09,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,ddos
46000,192.168.1.195,192.168.1.15,58532,23,6,1.749750e+09,0.002905,34424.688116,688.493762,344.246881,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


## Data Cleaning

### Removing useless features

Some columns were discarded because they did not provide useful information for training, or because they biased the information:
- Timestamp

In [7]:
removed_features: dict[str, list[str]] = {}

print("Before:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")

USELESS_FEATURES = ["timestamp"]#, "src_ip", "dst_ip"]
actually_dropped = [c for c in USELESS_FEATURES if c in df.columns]
df = df.drop(columns=actually_dropped)
removed_features["useless_features"] = actually_dropped

print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")

Before:
  Shape: (62133, 83)
  Columns: ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s', 'bwd_pkts_s', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts', 'totlen_bwd_pkts', 'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'bwd_pkt_len_max', 'bwd_pkt_len_min', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'pkt_len_max', 'pkt_len_min', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fwd_header_len', 'bwd_header_len', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'flow_iat_mean', 'flow_iat_max', 'flow_iat_min', 'flow_iat_std', 'fwd_iat_tot', 'fwd_iat_max', 'fwd_iat_min', 'fwd_iat_mean', 'fwd_iat_std', 'bwd_iat_tot', 'bwd_iat_max', 'bwd_iat_min', 'bwd_iat_mean', 'bwd_iat_std', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fin_flag_cnt', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'ece_flag_cnt', 'down_up_ratio', 'pkt_size_avg', 'fwd_seg

### Removing spaces from feature names

In [8]:
spaces_before = [c for c in df.columns if c != c.strip()]
print("Before - columns with spaces:", spaces_before)

df.columns = df.columns.str.strip()

spaces_after = [c for c in df.columns if c != c.strip()]
print("After  - columns with spaces:", spaces_after)
df.head()

Before - columns with spaces: []
After  - columns with spaces: []


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
49942,192.168.1.1,192.168.1.255,45898,20002,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,mitm
54998,192.168.1.100,192.168.1.195,60992,4280,6,0.036475,163453.607133,383.822627,191.911313,191.911313,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,web
43435,192.168.1.193,192.168.1.80,35586,23,6,0.003030,33000.031471,660.000629,330.000315,330.000315,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware
22839,192.168.1.102,192.168.1.19,2444,1883,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,ddos
46000,192.168.1.195,192.168.1.15,58532,23,6,0.002905,34424.688116,688.493762,344.246881,344.246881,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


### Removing rows with inoperative values

In [9]:
numeric_df = df.select_dtypes(include="number")

print("Before:")
print(f"  Shape: {df.shape}")
print(f"  NaN count:  {df.isna().sum().sum()}")
print(f"  Inf count:  {np.isinf(numeric_df).sum().sum()}")

df = df.replace([np.inf, -np.inf], np.nan).dropna()

numeric_df = df.select_dtypes(include="number")
print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  NaN count:  {df.isna().sum().sum()}")
print(f"  Inf count:  {np.isinf(numeric_df).sum().sum()}")
df.head()

Before:
  Shape: (62133, 82)
  NaN count:  0
  Inf count:  0

After:
  Shape: (62133, 82)
  NaN count:  0
  Inf count:  0


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
49942,192.168.1.1,192.168.1.255,45898,20002,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,mitm
54998,192.168.1.100,192.168.1.195,60992,4280,6,0.036475,163453.607133,383.822627,191.911313,191.911313,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,web
43435,192.168.1.193,192.168.1.80,35586,23,6,0.003030,33000.031471,660.000629,330.000315,330.000315,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware
22839,192.168.1.102,192.168.1.19,2444,1883,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,ddos
46000,192.168.1.195,192.168.1.15,58532,23,6,0.002905,34424.688116,688.493762,344.246881,344.246881,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


### Removing duplicate rows

In [10]:
print("Before:")
print(f"  Shape: {df.shape}")
print(f"  Duplicate rows: {df.duplicated().sum()}")

df = df.drop_duplicates()

print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  Duplicate rows: {df.duplicated().sum()}")
df.head()

Before:
  Shape: (62133, 82)
  Duplicate rows: 731

After:
  Shape: (61402, 82)
  Duplicate rows: 0


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
49942,192.168.1.1,192.168.1.255,45898,20002,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,mitm
54998,192.168.1.100,192.168.1.195,60992,4280,6,0.036475,163453.607133,383.822627,191.911313,191.911313,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,web
43435,192.168.1.193,192.168.1.80,35586,23,6,0.003030,33000.031471,660.000629,330.000315,330.000315,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware
22839,192.168.1.102,192.168.1.19,2444,1883,6,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,ddos
46000,192.168.1.195,192.168.1.15,58532,23,6,0.002905,34424.688116,688.493762,344.246881,344.246881,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


### Removing highly correlated features

In [11]:
CORR_THRESHOLD = 0.95

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Before:")
print(f"  Shape: {df.shape}")

corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
print(f"  Dropped (corr > {CORR_THRESHOLD}): {to_drop}")

df = df.drop(columns=to_drop)
removed_features["high_correlation"] = to_drop

print("\nAfter:")
print(f"  Shape: {df.shape}")
df.head()

Before:
  Shape: (61402, 82)
  Dropped (corr > 0.95): ['flow_pkts_s', 'fwd_pkts_s', 'tot_bwd_pkts', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'pkt_len_min', 'pkt_len_var', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'flow_iat_std', 'fwd_iat_tot', 'fwd_iat_max', 'fwd_iat_std', 'bwd_iat_mean', 'fwd_psh_flags', 'psh_flag_cnt', 'ack_flag_cnt', 'pkt_size_avg', 'fwd_seg_size_avg', 'bwd_seg_size_avg', 'active_mean', 'active_std', 'idle_max', 'idle_min', 'idle_mean']

After:
  Shape: (61402, 57)


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,bwd_pkts_s,tot_fwd_pkts,totlen_fwd_pkts,...,subflow_fwd_byts,subflow_bwd_pkts,subflow_bwd_byts,init_fwd_win_byts,init_bwd_win_byts,active_max,active_min,idle_std,cwr_flag_count,label
49942,192.168.1.1,192.168.1.255,45898,20002,17,0.000000,0.000000,0.000000,1,395,...,395,0,0,-1,-1,0.0,0.0,0.0,0,mitm
54998,192.168.1.100,192.168.1.195,60992,4280,6,0.036475,163453.607133,191.911313,7,943,...,943,7,5019,64240,65160,0.0,0.0,0.0,0,web
43435,192.168.1.193,192.168.1.80,35586,23,6,0.003030,33000.031471,330.000315,1,60,...,60,1,40,64240,4380,0.0,0.0,0.0,0,malware
22839,192.168.1.102,192.168.1.19,2444,1883,6,0.000000,0.000000,0.000000,1,40,...,40,0,0,512,-1,0.0,0.0,0.0,0,ddos
46000,192.168.1.195,192.168.1.15,58532,23,6,0.002905,34424.688116,344.246881,1,60,...,60,1,40,64240,5744,0.0,0.0,0.0,0,malware


### Removing near-zero variance features

In [12]:
VARIANCE_THRESHOLD = 1e-4

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Before:")
print(f"  Shape: {df.shape}")

variances = df[numeric_cols].var()
low_var_cols = variances[variances < VARIANCE_THRESHOLD].index.tolist()
print(f"  Dropped (var < {VARIANCE_THRESHOLD}): {low_var_cols}")

df = df.drop(columns=low_var_cols)
removed_features["near_zero_variance"] = low_var_cols

print("\nAfter:")
print(f"  Shape: {df.shape}")
df.head()

Before:
  Shape: (61402, 57)
  Dropped (var < 0.0001): ['fwd_urg_flags', 'bwd_urg_flags', 'urg_flag_cnt']

After:
  Shape: (61402, 54)


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,bwd_pkts_s,tot_fwd_pkts,totlen_fwd_pkts,...,subflow_fwd_byts,subflow_bwd_pkts,subflow_bwd_byts,init_fwd_win_byts,init_bwd_win_byts,active_max,active_min,idle_std,cwr_flag_count,label
49942,192.168.1.1,192.168.1.255,45898,20002,17,0.000000,0.000000,0.000000,1,395,...,395,0,0,-1,-1,0.0,0.0,0.0,0,mitm
54998,192.168.1.100,192.168.1.195,60992,4280,6,0.036475,163453.607133,191.911313,7,943,...,943,7,5019,64240,65160,0.0,0.0,0.0,0,web
43435,192.168.1.193,192.168.1.80,35586,23,6,0.003030,33000.031471,330.000315,1,60,...,60,1,40,64240,4380,0.0,0.0,0.0,0,malware
22839,192.168.1.102,192.168.1.19,2444,1883,6,0.000000,0.000000,0.000000,1,40,...,40,0,0,512,-1,0.0,0.0,0.0,0,ddos
46000,192.168.1.195,192.168.1.15,58532,23,6,0.002905,34424.688116,344.246881,1,60,...,60,1,40,64240,5744,0.0,0.0,0.0,0,malware


In [13]:
final_features = df.columns.tolist()
all_removed    = [col for cols in removed_features.values() for col in cols]

sep  = "=" * 60
sep2 = "-" * 60
lines = [
    sep,
    "FEATURE REPORT — DATA CLEANING SUMMARY",
    sep,
    "",
    f"Final features used ({len(final_features)}):",
]
for feat in sorted(final_features):
    lines.append(f"  + {feat}")

lines += ["", f"Excluded features ({len(all_removed)}):"]
for reason, cols in removed_features.items():
    if cols:
        lines += ["", f"  [{reason}]"]
        for col in sorted(cols):
            lines.append(f"    - {col}")

report = "\n".join(lines) + "\n"
print(report)

os.makedirs("../docs", exist_ok=True)
with open("../docs/features_report.txt", "w") as fh:
    fh.write(report)

print("Saved → ../docs/features_report.txt")

FEATURE REPORT — DATA CLEANING SUMMARY

Final features used (54):
  + active_max
  + active_min
  + bwd_blk_rate_avg
  + bwd_byts_b_avg
  + bwd_header_len
  + bwd_iat_max
  + bwd_iat_min
  + bwd_iat_std
  + bwd_iat_tot
  + bwd_pkt_len_max
  + bwd_pkt_len_min
  + bwd_pkts_b_avg
  + bwd_pkts_s
  + bwd_psh_flags
  + cwr_flag_count
  + down_up_ratio
  + dst_ip
  + dst_port
  + ece_flag_cnt
  + fin_flag_cnt
  + flow_byts_s
  + flow_duration
  + flow_iat_max
  + flow_iat_mean
  + flow_iat_min
  + fwd_blk_rate_avg
  + fwd_byts_b_avg
  + fwd_header_len
  + fwd_iat_mean
  + fwd_iat_min
  + fwd_pkt_len_max
  + fwd_pkt_len_mean
  + fwd_pkt_len_min
  + fwd_pkt_len_std
  + fwd_pkts_b_avg
  + idle_std
  + init_bwd_win_byts
  + init_fwd_win_byts
  + label
  + pkt_len_max
  + pkt_len_mean
  + pkt_len_std
  + protocol
  + rst_flag_cnt
  + src_ip
  + src_port
  + subflow_bwd_byts
  + subflow_bwd_pkts
  + subflow_fwd_byts
  + subflow_fwd_pkts
  + syn_flag_cnt
  + tot_fwd_pkts
  + totlen_bwd_pkts
  + totl

# Phase 1 — Binary Classification (Benign vs Malicious)

In [16]:
y = df["label"].apply(lambda x: "benign" if x == "benign" else "attack")
X = df.drop(columns=df.select_dtypes(exclude="number").columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

A prioridade aqui é o **recall**, para minimizar falsos negativos — é preferível suspeitar de tráfego benigno do que deixar um ataque passar como benigno.

In [17]:
def objective(trial):
    threshold = trial.suggest_float("threshold", 0.01, 0.5)

    params = {
        "objective": "multiclass",
        "num_class": y_train.nunique(),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "random_state": RANDOM_SEED,
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_bin": trial.suggest_int("max_bin", 128, 512),
        "class_weight": "balanced",
        "verbosity": -1,
        "force_row_wise": True,
        "n_jobs": 3,
    }

    model = lgb.LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_val)

        attack_idx = list(model.classes_).index("attack")
        y_pred = (probs[:, attack_idx] > threshold).astype(int)
        y_val_bin = (y_val == "attack").astype(int)

        scores.append(recall_score(y_val_bin, y_pred))

    return np.mean(scores)

In [18]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, timeout=1800, show_progress_bar=True)

[I 2026-04-29 08:53:19,390] A new study created in memory with name: no-name-536044d7-91ee-4d68-837e-115bec9333e8
Best trial: 0. Best value: 0.999976:   5%|▌         | 1/20 [00:02<00:44,  2.32s/it, 2.32/1800 seconds]

[I 2026-04-29 08:53:21,708] Trial 0 finished with value: 0.9999760226346329 and parameters: {'threshold': 0.23155884341505586, 'num_leaves': 85, 'max_depth': 1, 'min_child_samples': 94, 'min_split_gain': 0.8024303179552811, 'learning_rate': 0.06830864832927823, 'n_estimators': 669, 'subsample': 0.9390909268433277, 'subsample_freq': 4, 'colsample_bytree': 0.7274215551027972, 'reg_alpha': 0.013911562233062828, 'reg_lambda': 0.17407430251905442, 'max_bin': 465}. Best is trial 0 with value: 0.9999760226346329.


Best trial: 0. Best value: 0.999976:  10%|█         | 2/20 [00:07<01:13,  4.08s/it, 7.63/1800 seconds]

[I 2026-04-29 08:53:27,021] Trial 1 finished with value: 0.9999280679038987 and parameters: {'threshold': 0.2479799048189014, 'num_leaves': 142, 'max_depth': 13, 'min_child_samples': 51, 'min_split_gain': 0.8441548408520299, 'learning_rate': 0.010940874103280726, 'n_estimators': 686, 'subsample': 0.9729848572351416, 'subsample_freq': 3, 'colsample_bytree': 0.6630764347489975, 'reg_alpha': 3.5825584146774556e-06, 'reg_lambda': 0.00015099521286922508, 'max_bin': 191}. Best is trial 0 with value: 0.9999760226346329.


Best trial: 0. Best value: 0.999976:  15%|█▌        | 3/20 [00:12<01:15,  4.44s/it, 12.50/1800 seconds]

...

[I 2026-04-29 08:54:15,548] Trial 17 finished with value: 0.9999760226346329 and parameters: {'threshold': 0.3169932611927265, 'num_leaves': 64, 'max_depth': 6, 'min_child_samples': 83, 'min_split_gain': 0.1900146470170727, 'learning_rate': 0.0076133623895576754, 'n_estimators': 595, 'subsample': 0.6707665211799391, 'subsample_freq': 3, 'colsample_bytree': 0.7234271652235772, 'reg_alpha': 1.0087856636634957e-06, 'reg_lambda': 3.2106799653373615e-05, 'max_bin': 470}. Best is trial 3 with value: 1.0.


Best trial: 3. Best value: 1:  95%|█████████▌| 19/20 [00:59<00:03,  3.21s/it, 59.27/1800 seconds]

[I 2026-04-29 08:54:18,660] Trial 18 finished with value: 1.0 and parameters: {'threshold': 0.15498159627360594, 'num_leaves': 117, 'max_depth': 16, 'min_child_samples': 100, 'min_split_gain': 0.38072950021630947, 'learning_rate': 0.012659964419003876, 'n_estimators': 466, 'subsample': 0.8730781935455008, 'subsample_freq': 2, 'colsample_bytree': 0.9227744665519565, 'reg_alpha': 5.406192182070582, 'reg_lambda': 0.4972398702767122, 'max_bin': 423}. Best is trial 3 with value: 1.0.


Best trial: 3. Best value: 1: 100%|██████████| 20/20 [01:01<00:00,  3.09s/it, 61.81/1800 seconds]

[I 2026-04-29 08:54:21,199] Trial 19 finished with value: 1.0 and parameters: {'threshold': 0.19342855085225896, 'num_leaves': 171, 'max_depth': 4, 'min_child_samples': 59, 'min_split_gain': 0.09186595642059767, 'learning_rate': 0.005004460338494767, 'n_estimators': 323, 'subsample': 0.7551609700211028, 'subsample_freq': 3, 'colsample_bytree': 0.7676745451155341, 'reg_alpha': 1.0263979135144383e-08, 'reg_lambda': 0.00800763394349956, 'max_bin': 488}. Best is trial 3 with value: 1.0.


In [19]:
print("Best recall:", study.best_value)
print("Best params:", study.best_params)

Best recall: 1.0
Best params: {'threshold': 0.06170039389775678, 'num_leaves': 122, 'max_depth': 16, 'min_child_samples': 42, 'min_split_gain': 0.04664637256108528, 'learning_rate': 0.008020503367640243, 'n_estimators': 453, 'subsample': 0.7107993356021212, 'subsample_freq': 2, 'colsample_bytree': 0.8361024650399298, 'reg_alpha': 2.8648795108995407e-08, 'reg_lambda': 5.475448995529648e-05, 'max_bin': 505}


In [20]:
best = study.best_params.copy()
threshold = best.pop("threshold")

model = lgb.LGBMClassifier(**best)

In [21]:
model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,122
,max_depth,16
,learning_rate,0.008020503367640243
,n_estimators,453
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.04664637256108528
,min_child_weight,0.001
,min_child_samples,42


In [22]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      attack       1.00      1.00      1.00     10427
      benign       1.00      1.00      1.00      2000

    accuracy                           1.00     12427
   macro avg       1.00      1.00      1.00     12427
weighted avg       1.00      1.00      1.00     12427



In [25]:
recall_score(y_test, y_pred, pos_label="attack")

0.9997122854128704

In [24]:
dump(model, "../models/binary_classifier.pkl")

['../models/binary_classifier.pkl']

# Phase 2 — Multiclassification

# Phase 3 — Clustering